In [12]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict
import uuid

In [13]:
# Step 1: Initialize Store
store = InMemoryStore()

In [14]:
# Step 2: Define Extended State
class AgentState(MessagesState):
    """State with user context for LTM retrieval"""
    user_id: str

In [15]:
# Step 3: LLM
llm = ChatOpenAI(model="gpt-4o-mini")

In [16]:
# Step 4: LTM Retrieval Node
def retrieve_user_facts(state: AgentState) -> dict:
    """
    Retrieve facts using semantic search.
    """
    user_id = state["user_id"]
    namespace = (user_id, "facts")
    
    last_message = state["messages"][-1].content if state["messages"] else ""
    
    try:
        # Semantic search for relevant facts
        results = store.search(
            namespace=namespace,
            query=last_message,  # Find facts similar to current message
            limit=5
        )
        
        facts_text = ""
        for result in results:
            facts_text += f"- {result.value}\n"
        
        return {"context": facts_text if facts_text else "No prior facts."}
    except Exception as e:
        return {"context": "No prior facts."}

In [17]:
# Step 5: Agent Node (uses LTM context)
def agent_with_ltm(state: AgentState) -> dict:
    """
    Agent using PostgreSQL-backed long-term memory.
    """
    user_id = state["user_id"]
    namespace = (user_id, "facts")
    
    # Retrieve relevant facts
    last_message = state["messages"][-1].content
    
    facts_text = "No prior facts."
    try:
        results = store.search(namespace, query=last_message, limit=5)
        if results:
            facts_text = "\n".join([f"- {r.value}" for r in results])
    except:
        pass
    
    # Build system message with facts
    system_msg = SystemMessage(content=f"""
You are a helpful assistant with access to user's long-term memory.

User's Known Facts:
{facts_text}

Use these facts to provide personalized and contextual responses.
""")
    
    print("facts_text", facts_text)
    messages = [system_msg] + state["messages"]
    response = llm.invoke(messages)
    
    return {"messages": [response]}

In [18]:
# Step 6: Memory Update Node
def update_user_facts(state: AgentState) -> dict:
    """
    Extract important facts from conversation and store in LTM.
    """
    user_id = state["user_id"]
    namespace = (user_id, "facts")
    
    # Get the last user message
    last_msg = None
    for msg in reversed(state["messages"]):
        if msg.type == "human":
            last_msg = msg.content
            break
    
    if not last_msg:
        return {}
    
    # Ask LLM to extract facts
    extraction_prompt = f"""
From this user message, extract 1-2 important facts about the user:
"{last_msg}"

Format: Just the facts, one per line. Examples:
- User works as a software engineer
- User is interested in machine learning

Extract facts:"""
    
    response = llm.invoke([
        {"role": "user", "content": extraction_prompt}
    ])
    
    facts = response.content.strip().split("\n")
    
    # Store facts in LTM
    for fact in facts:
        if fact.strip():
            fact_id = str(uuid.uuid4())
            store.put(namespace, fact_id, fact.strip())
    
    return {}

In [19]:
# Step 7: Build Graph
def create_ltm_agent():
    builder = StateGraph(AgentState)
    
    builder.add_node("agent", agent_with_ltm)
    builder.add_node("update_ltm", update_user_facts)
    
    builder.add_edge(START, "agent")
    builder.add_edge("agent", "update_ltm")
    builder.add_edge("update_ltm", END)
    
    graph = builder.compile(store=store)
    
    return graph

In [ ]:
# Step 8: Usage
def main():
    graph = create_ltm_agent()
    
    user_id = "user_alice"
    config = {"configurable": {"thread_id": "thread-1"}}
    
    # Turn 1
    print("=== Turn 1 ===")
    result = graph.invoke(
        {
            "messages": [HumanMessage(content="Hi! I'm a data scientist interested in Python and machine learning")],
            "user_id": user_id
        },
        config
    )
    print(f"Bot: {result['messages'][-1].content}\n")
    
    # Turn 2 (same thread)
    print("=== Turn 2 (same thread) ===")
    result = graph.invoke(
        {
            "messages": [HumanMessage(content="What should I focus on learning next?")],
            "user_id": user_id
        },
        config
    )
    print(f"Bot: {result['messages'][-1].content}\n")
    
    # Turn 3 (different thread, same user)
    print("=== Turn 3 (different thread, same user) ===")
    config2 = {"configurable": {"thread_id": "thread-2"}}
    
    result = graph.invoke(
        {
            "messages": [HumanMessage(content="Can you recommend some projects?")],
            "user_id": user_id
        },
        config2
    )
    print(f"Bot: {result['messages'][-1].content}\n")

In [21]:
if __name__ == "__main__":
    main()

=== Turn 1 ===
facts_text No prior facts.
Bot: That's great! Python is a powerful programming language for data science, and there are plenty of libraries available, such as NumPy, pandas, scikit-learn, and TensorFlow, to help with machine learning. Do you have any specific projects or areas in machine learning that you're interested in exploring?

=== Turn 2 (same thread) ===
facts_text - - User is a data scientist
- - User is interested in Python and machine learning
Bot: Given your background as a data scientist and your interest in Python and machine learning, here are a few areas you might consider focusing on next:

1. **Deep Learning**: If you haven't explored neural networks in depth, this could be a great addition. Consider learning frameworks like TensorFlow or PyTorch.

2. **Natural Language Processing (NLP)**: With the growth of text data, understanding NLP techniques and libraries like NLTK and spaCy can be highly beneficial.

3. **Reinforcement Learning**: This is an exci